In [1]:
import sys
from pathlib import Path
def _find_project_root():
    """
    Sube directorios desde el cwd hasta encontrar una carpeta 'src'.
    Devuelve la carpeta raíz del proyecto (la que contiene 'src').
    """
    p = Path.cwd().resolve()
    for _ in range(5):  # sube hasta 5 niveles por seguridad
        if (p / "scraper_ICBC").exists():
            return p
        p = p.parent
    raise RuntimeError("No se encontró carpeta 'src' hacia arriba desde el notebook.")

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))  # 👈 agregamos la RAÍZ, no 'src'
print(PROJECT_ROOT)

C:\ctcCatalogMonitor


In [2]:
from __future__ import annotations
import time
from datetime import datetime
import pandas as pd
from common.config import (
    COMPARE_MAPPINGS,
    PICKLE_DIR,
    SCRAPER_CATALOG_MAPPINGS,
    LOG_ALL,
    LOG_PIPELINE,
    PURGE_OLD_FILES,
    CSV_DIR,
    SCRAPER_KEY_COLUMNS
)
from common.storage import write_log, save_csv, purge_all_old, load_newest_csv, load_newest_pickle
from validator.src.comparer import compare_catalogs

In [6]:
#busco cvs de un scraper
scraper_name = "scraper_GoPoint_UY"
pattern_csv = f"*{scraper_name.upper().split('_')[-1]}*.csv"  # ej: *ICBC*.csv
df_web = load_newest_csv(CSV_DIR, pattern_csv)
df_web.head()

,idProdutoFornecedor,nome,descricao,imagens,habilitado,publicado,valor_min_reais,endpoint_name
0,BUQUEB0030,- - 1 PASAJE Ida y Vueta- Montevideo a Bueno...,<h3>Detalle</h3><p>1 PASAJE a BUENOS AIRES (VI...,['https://ctc-productos.s3.amazonaws.com/buque...,True,True,5696.0,latam_uy_migc_all
1,GVTONE001UY,Tonel Privado - Tonel Privado - Gift Card Virt...,<h3>Detalle</h3><p>La Gift Card Virtual podrá ...,['https://ctc-productos.s3.amazonaws.com/gvton...,True,True,380.0,latam_uy_migc_all
2,LAESTE003,La Esteña - La Esteña - Gift Card Virtual USD 100,<h3>Detalle</h3><p>La Gift Card Virtual que em...,['https://ctc-productos.s3.amazonaws.com/laest...,True,True,3971.0,latam_uy_migc_all
3,LAESTE001,La Esteña - La Esteña - Gift Card Virtual USD 20,<h3>Detalle</h3><p>La Gift Card Virtual que em...,['https://ctc-productos.s3.amazonaws.com/laest...,True,True,752.0,latam_uy_migc_all
4,PANTAI003,PANTHAI - PANTHAI - Gift Card Virtual $ 5.000,<h3>Detalle</h3><p>La Gift Card Virtual que em...,['https://ctc-productos.s3.amazonaws.com/panta...,True,True,5225.0,latam_uy_migc_all


In [6]:
df_db = load_newest_pickle(PICKLE_DIR, pattern="*_products_expected.p")
df_db.value_counts("CatalogoId")

CatalogoId
78    725
82    616
77    583
76    541
83     79
Name: count, dtype: int64

In [7]:

catalog_ids = SCRAPER_CATALOG_MAPPINGS.get(scraper_name, [])
display(catalog_ids)
print(catalog_ids and "CatalogoId" in df_db.columns)



[78, 83]

True


In [9]:

if catalog_ids and "CatalogId" in df_db.columns:
    df_db = df_db[df_db["CatalogId"].isin(catalog_ids)]
print(df_db[df_db["CatalogoId"] == 78])

      CatalogoId         sku                                     nombre  \
22            78  EXBIUY0001                                Box Bonjour   
23            78  EXBIUY0002                              Box Take Away   
24            78  EXBIUY0003                         Box Cocktail Night   
25            78  EXBIUY0004                          Box Petit Gourmet   
26            78  EXBIUY0005                                Box Bodegas   
...          ...         ...                                        ...   
2521          78  ENCATEX608        PLAYMOBIL CUATRICICLO POLICIA 71092   
2522          78  ENCATEX609  PLAYMOBIL DISNEY MINNIE EN LA PLAYA 71706   
2523          78  ENCATEX610                      PLAYMOBIL VESPA 71622   
2524          78  ENCATEX611  PLAYMOBIL DISNEY MICKEY EN LA PLAYA 71707   
2525          78  ENCATEX612       PLAYMOBIL DISNEY MICKEY COHETE 71771   

                                         ArtDescription  PrecioLista  \
22    Box Bonjour: Regalá e

In [ ]:

key_web, key_db = SCRAPER_KEY_COLUMNS.get("scraper_icbc", ("sku", "sku"))
print(key_web)
print(key_db)

reference
sku


In [13]:
df_db.head()
result = compare_catalogs(df_db, df_web, key_col_db=key_db, key_col_web=key_web, scraper_name=scraper_name)
print(result)

{'missing': 1240, 'extra': 0, 'matched': 783, 'diffs': 0}


In [9]:
df_db[df_db["sku"] == "GVMIGCPAPAII08"]

,CatalogoId,sku,nombre,ArtDescription,PrecioLista,PrecioVenta,Cantidad
1534,76,GVMIGCPAPAII08,Gift Card Virtual $ 500.000,!Las mejores marcas para Papá en una sola tarj...,0.0,500000.0,100


In [16]:

db_keys = set(df_db[key_db].astype(str))
web_keys = set(df_web[key_web].astype(str))
common_keys = db_keys & web_keys
print(result)


{'missing': 1240, 'extra': 0, 'matched': 783, 'diffs': 0}


In [19]:
print(result["missing"])

1240


In [20]:
from validator.src.pipeline import run_pipeline_for_channel

run_pipeline_for_channel("scraper_GoPoint_UY")

[ERROR] No se encontraron archivos que coincidan con *scraper_GoPoint_UY*.csv en C:\ctcCatalogMonitor\data\outputs\csv
✅ Ejecución finalizada.


{'scraper': 'scraper_GoPoint_UY',
 'missing': 0,
 'extra': 0,
 'diffs': 0,
 'status': 'ERROR: No se encontraron archivos que coincidan con *scraper_GoPoint_UY*.csv en C:\\ctcCatalogMonitor\\data\\outputs\\csv'}

In [3]:
from validator.src.pipeline import run_full_pipeline

run_full_pipeline(verbose= True)

✅ Ejecución finalizada.
✅ Ejecución finalizada.
✅ Ejecución finalizada.
===== Resumen global =====
scraper_icbc          missing=436   extra=0     matched=105   diffs=0      OK
scraper_GoPoint_AR    missing=113   extra=0     matched=1082  diffs=0      OK
scraper_GoPoint_UY    missing=18    extra=0     matched=784   diffs=0      OK


[{'scraper': 'scraper_icbc',
  'missing': 436,
  'extra': 0,
  'matched': 105,
  'diffs': 0,
  'status': 'OK'},
 {'scraper': 'scraper_GoPoint_AR',
  'missing': 113,
  'extra': 0,
  'matched': 1082,
  'diffs': 0,
  'status': 'OK'},
 {'scraper': 'scraper_GoPoint_UY',
  'missing': 18,
  'extra': 0,
  'matched': 784,
  'diffs': 0,
  'status': 'OK'}]